# OnionEye · 02 · Train the finder (onion + coin)
**Runtime → Change runtime type → T4 GPU**, then Run all. About 2–3 h. If Colab disconnects, just Run all again: it resumes from the last epoch (saved in your Drive).

Output: `MyDrive/onioneye/models/finder.onnx` and `finder_best.pt`, plus scores in `reports/`.

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim

In [ ]:
# ---- setup: works on Google Colab (default) and Kaggle
import os, sys, glob, shutil, zipfile, random
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive/onioneye', '/content'
elif IN_KAGGLE:
    ROOT, WORK = '/kaggle/working/onioneye', '/kaggle/temp'
else:
    ROOT, WORK = os.path.abspath('onioneye_data'), os.path.abspath('work')
for d in (ROOT, WORK, f'{ROOT}/models', f'{ROOT}/reports'):
    os.makedirs(d, exist_ok=True)
print('saving to', ROOT)

In [ ]:
FINDER = f'{WORK}/finder'
if not os.path.exists(f'{FINDER}/data.yaml'):
    zipfile.ZipFile(f'{ROOT}/finder_dataset.zip').extractall(FINDER)
import yaml
for name in ['data.yaml', 'data_unseen.yaml']:
    cfg = yaml.safe_load(open(f'{FINDER}/{name}')); cfg['path'] = FINDER
    yaml.safe_dump(cfg, open(f'{FINDER}/{name}', 'w'), sort_keys=False)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Train YOLO11n-seg

In [ ]:
from ultralytics import YOLO
RUNS = f'{ROOT}/runs'
last = f'{RUNS}/finder/weights/last.pt'
if os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11n-seg.pt').train(
        data=f'{FINDER}/data.yaml', imgsz=640, epochs=60, patience=15, batch=16, workers=4,
        project=RUNS, name='finder', exist_ok=True, seed=42, cos_lr=True,
        fliplr=0.5, flipud=0.5, degrees=15, hsv_h=0.01, close_mosaic=10)

## 2 · Scores: own test split, and a different author's photos (onionthesis1)

In [ ]:
import pandas as pd
best = YOLO(f'{RUNS}/finder/weights/best.pt')
rows = []
for label, cfg in [('test (same sources)', 'data.yaml'), ('unseen onionthesis1', 'data_unseen.yaml')]:
    m = best.val(data=f'{FINDER}/{cfg}', split='test', imgsz=640, plots=False)
    rows.append({'set': label, 'box mAP50': m.box.map50, 'mask mAP50': m.seg.map50,
                 'mask mAP50-95': m.seg.map, 'precision': m.box.mp, 'recall': m.box.mr})
df = pd.DataFrame(rows).round(3); display(df)
df.to_csv(f'{ROOT}/reports/finder_scores.csv', index=False)

## 3 · Predictions on 8 unseen photos

In [ ]:
import matplotlib.pyplot as plt
imgs = random.sample(glob.glob(f'{FINDER}/images/test_unseen/*') or glob.glob(f'{FINDER}/images/test/*'), 8)
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for ax, r in zip(axs.flat, best.predict(imgs, imgsz=640, conf=0.4, verbose=False)):
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{ROOT}/reports/finder_predictions.png', dpi=70); plt.show()

## 4 · Export for the backend

In [ ]:
shutil.copy(f'{RUNS}/finder/weights/best.pt', f'{ROOT}/models/finder_best.pt')
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
shutil.copy(onnx_path, f'{ROOT}/models/finder.onnx')
import onnxruntime as ort
s = ort.InferenceSession(onnx_path)
print([(o.name, o.shape) for o in s.get_outputs()])          # expect (1, 38, 8400) and mask protos
print('classes:', s.get_modelmeta().custom_metadata_map.get('names'))

Done ✅ Download `models/finder.onnx` → put it in `backend/models/`. Next: **03_build_defect_data**.